# 00 — Citi MQA Replication on Real Data

**Phase 1 deliverable · Commodity Hedging & Structuring Workbench**

This notebook reproduces the Forage Citi MQA Task 3 pricing exercise (toy inputs) beside the **real frozen coffee curve** (`data/frozen/`, SHA-256-manifested, gated). Goal: see exactly where the simulation's numbers diverge from the market.

*Personal learning project extending the Citi MQA Forage simulation — not affiliated with or endorsed by Citi. Data: Yahoo Finance (unofficial source, accepted and documented; gates fail closed to gold `GC=F` if coffee fails).*

## 1. Frozen data + gates

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
from scipy.stats import norm

from hedging_workbench.data.download import FROZEN_DIR, load_frozen, verify_manifest
from hedging_workbench.data.gates import evaluate
from hedging_workbench.data.universe import UNIVERSES

assert not verify_manifest(FROZEN_DIR, "coffee"), "coffee manifest tampered"
coffee = load_frozen(list(UNIVERSES["coffee"]))
gold = load_frozen(list(UNIVERSES["gold"]))
report = evaluate(coffee, gold)
print(report.summary())
pd.DataFrame(report.series).T

[PASS] coffee: 9 series, 0 failures


,rows,last_close,last_date
KC=F,674,324.25,2026-09-04
KCU26.NYB,675,324.25,2026-09-04
KCZ26.NYB,675,295.600006,2026-09-04
KCH27.NYB,613,287.399994,2026-09-04
KCK27.NYB,569,285.049988,2026-09-04
KCN27.NYB,529,283.399994,2026-09-04
KCU27.NYB,486,281.450012,2026-09-04
KCZ27.NYB,423,278.75,2026-09-04
KCH28.NYB,361,276.899994,2026-09-04


**Curve snapshot (2026-09-04).** The real chain is in **backwardation**: front `KC=F`/Sep-26 at 324.25 ¢/lb, Dec-26 at 295.60, gently declining out to Mar-28 at 276.90. The Forage sim assumed contango (F > S via positive carry). Both facts drive everything below.

## 2. Toy reproduction — Forage Task 3 exactly as given

In [2]:
# Task 3 inputs (toy)
S, r, d, T, sigma, K = 1.20, 0.02, 0.01, 0.5, 0.25, 1.25

F_toy = S * np.exp((r + d) * T)

def black76(F, K, T, r, sigma):
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return np.exp(-r * T) * (F * norm.cdf(d1) - K * norm.cdf(d2))

c_toy = black76(F_toy, K, T, r, sigma)

# MC validation of the toy call (risk-neutral GBM on the futures, 10k paths, seed 42)
rng = np.random.default_rng(42)
n, steps = 10_000, 126
dt = T / steps
FT = F_toy * np.exp(np.cumsum(
    (-0.5 * sigma**2 * dt + sigma * np.sqrt(dt) * rng.standard_normal((n, steps))), axis=1))[:, -1]
payoffs = np.maximum(FT - K, 0)
c_mc = np.exp(-r * T) * payoffs.mean()
se = np.exp(-r * T) * payoffs.std() / np.sqrt(n)

print(f"Toy futures fair value      F = {F_toy:.4f} $/lb   (sim answer: 1.218)")
print(f"Toy Black-76 ATM call       c = {c_toy:.4f} $/lb   (sim answer: 0.071)")
print(f"Toy MC (10k paths)          c = {c_mc:.4f} +/- {se:.4f} $/lb   within 1 SE of closed form: {abs(c_mc - c_toy) < se}")

Toy futures fair value      F = 1.2181 $/lb   (sim answer: 1.218)
Toy Black-76 ATM call       c = 0.0712 $/lb   (sim answer: 0.071)
Toy MC (10k paths)          c = 0.0705 +/- 0.0013 $/lb   within 1 SE of closed form: True


## 3. Real data — the same pricing machinery on the frozen curve

In [3]:
spot = coffee["KC=F"].iloc[-1] / 100          # continuous front, $/lb
kcu26 = coffee["KCU26.NYB"].iloc[-1] / 100    # Sep-26 contract
kcz26 = coffee["KCZ26.NYB"].iloc[-1] / 100    # Dec-26 contract

# Same Black-76 call as the toy, but on the real front price.
# Strike scaled 2.5x with the spot regime so moneyness matches the toy ATM.
F_real = kcu26
c_real = black76(F_real, K * 2.5, T, r, sigma)

cmp = pd.DataFrame(
    {"toy (sim)": [S, c_toy], "real (frozen)": [spot, c_real]},
    index=["front price ($/lb)", "ATM call ($/lb)"])
cmp["real / toy"] = cmp["real (frozen)"] / cmp["toy (sim)"]
display(cmp.round(4))
print(f"real Dec-26 futures {kcz26:.4f} vs toy forward {F_toy:.4f}"
      f" -> x{kcz26/F_toy:.2f}, and the real curve DECAYS outward, not carries")

,toy (sim),real (frozen),real / toy
front price ($/lb),1.2000,3.2425,2.7021
ATM call ($/lb),0.0712,0.2850,4.0029


real Dec-26 futures 2.9560 vs toy forward 1.2181 -> x2.43, and the real curve DECAYS outward, not carries


In [4]:
# Teaser for Phase 2: implied convenience yield y from the real curve
# F2 = F1 * exp((r + d - y) * dt),  dt = 0.25y (Sep-26 -> Dec-26)
from math import log
y_impl = r + d - log(kcz26 / kcu26) / 0.25
print(f"implied annualised convenience yield Sep-26 -> Dec-26: {y_impl:+.1%}")
print("toy assumed y = 0 (abundant inventories); the real market effectively PAYS"
      f" holders ~{abs(y_impl):.0%}/yr to store coffee -> scarcity, not abundance")

implied annualised convenience yield Sep-26 -> Dec-26: +40.0%
toy assumed y = 0 (abundant inventories); the real market effectively PAYS holders ~40%/yr to store coffee -> scarcity, not abundance


## 4. Where the sim diverges — and why it matters

| Dimension | Forage toy | Real frozen curve | Consequence |
|---|---|---|---|
| Price level | $1.20/lb | $3.24/lb | every notional, margin, and VaR number scales ~2.7x |
| Curve shape | contango (+3% carry) | backwardation (Dec-26 8.8% *below* front) | a roaster's long hedge now benefits from roll instead of bleeding roll cost |
| Convenience yield | assumed 0 | implied strongly positive | cost-of-carry fair value sits *below* spot; the Task 3 forward logic inverts |
| Vol | given 25% | unknown (no free coffee options) | Phase 1 keeps toy vol; Phase 2 calibrates EWMA/GARCH on real returns |

**Learning:** the Task 3–5 arc survives contact with real data, but every *number* changes. The hedge design question (Phase 3) — collar strikes, month selection — must be answered against this backwardated curve, not the sim's contango. That is the whole reason freezes real data before any modelling.

## Reproducibility

- Frozen snapshots: `data/frozen/*.csv` + `manifest_coffee.json` / `manifest_gold.json` (SHA-256 per file, verified above)
- Gates: `python -m hedging_workbench.data.gates` (coffee primary, gold fallback, fail closed)
- Tests: `pytest` — 6 tests: gate pass/fail, fallback trigger, checksum tamper
- Limitations: unofficial Yahoo source; continuous `KC=F` used as spot proxy; toy vol carried forward (no free coffee options data); frozen 2026-09-04